# PixAI Core API · اجرای مدل چت

این دفترچه مدل پایه Qwen3-1.7B را در Runtime ابری Colab اجرا می‌کند و یک API موقت برای رابط GitHub Pages می‌سازد. اجرای آن دائمی نیست؛ با قطع Runtime باید دوباره اجرا شود.

**وضعیت مدل:** این مدل پایه است و هنوز با دیتاست AIPix آموزش اختصاصی ندیده است. این مرحله فقط اتصال و اجرای مدل را فراهم می‌کند.

In [ ]:
%pip -q install -U transformers accelerate fastapi uvicorn


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen3-1.7B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
load_kwargs = {'device_map': 'auto'}
if torch.cuda.is_available():
    load_kwargs['torch_dtype'] = torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()
print('مدل بارگذاری شد:', MODEL_ID)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'در دسترس نیست؛ CPU ممکن است بسیار کند باشد')


In [ ]:
SYSTEM_PROMPT = '''تو PixAI هستی، دستیار هوش مصنوعی فارسی‌زبان پروژه AIPix.
- به زبان کاربر پاسخ بده؛ فارسی را روان، خودمانی و روشن بنویس.
- پاسخ‌ها دقیق، محترمانه و متناسب با سؤال باشند.
- اگر پرسیدند کی هستی، بگو دستیار هوش مصنوعی پروژه AIPix هستی.
- اگر پاسخ را نمی‌دانی، صادقانه بگو و حدس را واقعیت جا نزن.
- ادعا نکن به اینترنت، فایل‌ها یا ابزارها وصل هستی مگر واقعاً فعال باشند.
- اگر درخواست تصویر شد، فقط در صورت فعال بودن ابزار تصویر ادعای ساخت تصویر کن.
- رمز یا اطلاعات محرمانه درخواست نکن.
- در موضوعات حساس با احتیاط و احترام پاسخ بده.
هدف تو کمک دقیق و امن به کاربر است.
'''


## راه‌اندازی API

این سلول یک نشانی موقت و یک کلید دسترسی تصادفی می‌سازد. نشانی و کلید را فقط در تنظیمات چت‌بات شخصی خودت وارد کن و آن‌ها را عمومی منتشر نکن. Runtime را تا پایان استفاده روشن نگه دار.

In [ ]:
import secrets, threading, time
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Optional
import uvicorn

API_TOKEN = secrets.token_urlsafe(32)
app = FastAPI(title='PixAI Core API')
app.add_middleware(CORSMiddleware, allow_origin_regex=r'https://.*\\.github\\.io', allow_credentials=False, allow_methods=['GET','POST','OPTIONS'], allow_headers=['Authorization','Content-Type'])

class ChatRequest(BaseModel):
    model: Optional[str] = 'pixai-qwen'
    messages: List[Dict[str, str]]
    max_tokens: Optional[int] = 320

@app.get('/health')
def health():
    return {'ok': True, 'model': MODEL_ID, 'status': 'ready'}

@app.post('/v1/chat/completions')
def chat(req: ChatRequest, authorization: Optional[str] = Header(default=None)):
    if authorization != f'Bearer {API_TOKEN}':
        raise HTTPException(status_code=401, detail='Invalid API token')
    safe = [{'role':'system','content':SYSTEM_PROMPT}]
    for item in req.messages[-16:]:
        role, content = item.get('role'), item.get('content')
        if role in ('user','assistant') and isinstance(content, str):
            safe.append({'role':role,'content':content[:4000]})
    prompt = tokenizer.apply_chat_template(safe, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=min(max(int(req.max_tokens or 320), 32), 600), do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.08, pad_token_id=tokenizer.eos_token_id)
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return {'choices':[{'message':{'role':'assistant','content':answer}}], 'model':MODEL_ID}

def run_api():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=run_api, daemon=True).start()
time.sleep(2)
from google.colab import output
PUBLIC_URL = output.eval_js('google.colab.kernel.proxyPort(8000)')
print('PixAI API endpoint:', PUBLIC_URL + '/v1/chat/completions')
print('PixAI API token:', API_TOKEN)
print('Health check:', PUBLIC_URL + '/health')
